# Bike Rental Demand Prediction — GUI in Google Colab

This notebook trains and compares regression models, then launches an interactive Gradio GUI directly inside Colab. It automatically fetches current weather using Open-Meteo.

**Problem type:** Regression, because `cnt` is a numeric rental count. Metrics: MAE, RMSE, and R².

In [ ]:
!pip -q install gradio requests scikit-learn pandas numpy matplotlib joblib

In [ ]:
import io, zipfile, urllib.request, warnings, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import gradio as gr
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
import joblib
warnings.filterwarnings("ignore")
RANDOM_STATE = 42


## 1. Load the Bike Sharing dataset

In [ ]:
# Download the UCI Bike Sharing Dataset hourly file (hour.csv)
url = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
response = requests.get(url, timeout=30)
response.raise_for_status()
zf = zipfile.ZipFile(io.BytesIO(response.content))
df = pd.read_csv(zf.open("hour.csv"))

print("Dataset shape:", df.shape)
display(df.head())
print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing values"))
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe().T)


## 2. Explore the target and hourly demand

In [ ]:
df = df.drop_duplicates().copy()
FEATURES = ["season", "yr", "mnth", "hr", "holiday", "weekday",
            "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]
TARGET = "cnt"
df = df[FEATURES + [TARGET]].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df["cnt"].hist(bins=30, ax=axes[0])
axes[0].set_title("Distribution of hourly bike rentals")
axes[0].set_xlabel("cnt")
df.groupby("hr")["cnt"].mean().plot(marker="o", ax=axes[1])
axes[1].set_title("Average rentals by hour")
axes[1].set_xlabel("Hour")
axes[1].set_ylabel("Average cnt")
plt.tight_layout()
plt.show()


## 3. Train and compare regression models

In [ ]:
X = df[FEATURES]
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

models = {
    "Linear Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Ridge Regression": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=150, random_state=RANDOM_STATE, n_jobs=-1, min_samples_leaf=2
        ))
    ]),
    "Hist Gradient Boosting": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_iter=150, learning_rate=0.08, random_state=RANDOM_STATE
        ))
    ])
}

rows = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2 Score": r2_score(y_test, pred)
    })

results_df = pd.DataFrame(rows).sort_values("RMSE").reset_index(drop=True)
display(results_df.round(3))
best_name = results_df.loc[0, "Model"]
best_model = models[best_name]
print("Best model by lowest test RMSE:", best_name)

os.makedirs("artifacts", exist_ok=True)
joblib.dump(best_model, "artifacts/best_bike_model.joblib")
joblib.dump(FEATURES, "artifacts/bike_features.joblib")
results_df.to_csv("artifacts/model_comparison.csv", index=False)


## 4. Weather and prediction helpers

Open-Meteo provides current weather without an API key. The weather features are converted approximately to the normalized scale used by the UCI dataset. This is a student-project prototype; current weather is not a historical forecast for past dates, and the mapping is approximate.

In [ ]:
def fetch_weather(city):
    city = (city or "").strip()
    if not city:
        raise ValueError("Enter a city name.")
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        timeout=20
    )
    geo.raise_for_status()
    places = geo.json().get("results", [])
    if not places:
        raise ValueError("City not found. Try a larger nearby city.")
    place = places[0]
    wx = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,weather_code,wind_speed_10m",
            "timezone": "auto"
        },
        timeout=20
    )
    wx.raise_for_status()
    return place, wx.json()["current"]

def weather_category(code, precipitation):
    # Approximate UCI weather category: 1 clear, 2 mist/cloudy, 3 light precipitation, 4 severe
    if precipitation >= 5 or code >= 95:
        return 4
    wet_codes = [51,53,55,56,57,61,63,65,66,67,71,73,75,77,80,81,82,85,86]
    if precipitation > 0 or code in wet_codes:
        return 3
    if code in [3,45,48]:
        return 2
    return 1

def predict_rentals(city, date, hour, season, holiday, workingday):
    try:
        place, wx = fetch_weather(city)
        d = pd.to_datetime(date).date()
        # UCI dataset uses weekday=0 for Sunday, whereas Python weekday=0 is Monday.
        weekday = (d.weekday() + 1) % 7
        row = {
            "season": int(season),
            "yr": 1,  # binary year indicator in the original dataset
            "mnth": d.month,
            "hr": int(hour),
            "holiday": int(holiday),
            "weekday": weekday,
            "workingday": int(workingday),
            "weathersit": weather_category(int(wx.get("weather_code", 0)), float(wx.get("precipitation", 0))),
            "temp": float(wx["temperature_2m"]) / 41.0,
            "atemp": float(wx["apparent_temperature"]) / 50.0,
            "hum": float(wx["relative_humidity_2m"]) / 100.0,
            "windspeed": float(wx["wind_speed_10m"]) / 67.0
        }
        input_df = pd.DataFrame([row], columns=FEATURES)
        estimate = max(0, float(best_model.predict(input_df)[0]))
        weather_summary = (
            f"### Current weather in {place['name']}, {place.get('country', '')}\n"
            f"- Temperature: **{wx['temperature_2m']} °C**\n"
            f"- Feels like: **{wx['apparent_temperature']} °C**\n"
            f"- Humidity: **{wx['relative_humidity_2m']}%**\n"
            f"- Wind speed: **{wx['wind_speed_10m']} km/h**\n"
            f"- Weather code: **{wx.get('weather_code', 'N/A')}**"
        )
        prediction_text = (
            f"## Estimated bike rentals: {round(estimate):,}\n"
            f"Model used: **{best_name}**\n\n"
            "This is an estimate, not a guaranteed count. Current weather and normalized feature mappings are approximate."
        )
        return weather_summary, prediction_text
    except Exception as e:
        return "### Weather lookup failed\nCheck the city name and internet connection, then try again.", f"**Error:** {e}"


## 5. Launch the GUI in Google Colab

In [ ]:
import datetime

with gr.Blocks(title="Bike Rental Demand Prediction") as demo:
    gr.Markdown("# 🚲 Bike Rental Demand Prediction")
    gr.Markdown("Enter a city to fetch current weather automatically, then estimate hourly bike rental demand.")
    with gr.Row():
        city_input = gr.Textbox(label="City", value="Nagpur", placeholder="e.g., Nagpur")
        date_input = gr.Textbox(label="Date (YYYY-MM-DD)", value=str(datetime.date.today()))
    with gr.Row():
        hour_input = gr.Slider(0, 23, value=datetime.datetime.now().hour, step=1, label="Hour of day")
        season_input = gr.Dropdown(
            choices=[("Spring", 1), ("Summer", 2), ("Fall", 3), ("Winter", 4)],
            value=2, label="Season (UCI dataset coding)"
        )
    with gr.Row():
        holiday_input = gr.Radio(choices=[("No", 0), ("Yes", 1)], value=0, label="Holiday?")
        workingday_input = gr.Radio(choices=[("No", 0), ("Yes", 1)], value=1, label="Working day?")
    run_button = gr.Button("Fetch Weather & Predict", variant="primary")
    weather_output = gr.Markdown()
    prediction_output = gr.Markdown()
    run_button.click(
        fn=predict_rentals,
        inputs=[city_input, date_input, hour_input, season_input, holiday_input, workingday_input],
        outputs=[weather_output, prediction_output]
    )

demo.launch(share=True, debug=False)


## 6. Download the trained artifacts

Run this cell if you want to download the trained model files.

In [ ]:
from google.colab import files
import shutil
shutil.make_archive("bike_model_artifacts", "zip", "artifacts")
files.download("bike_model_artifacts.zip")


## References
- UCI Machine Learning Repository, Bike Sharing Dataset: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset
- Open-Meteo API: https://open-meteo.com/
- Scikit-learn documentation: https://scikit-learn.org/stable/
- Gradio documentation: https://www.gradio.app/docs